# SpecDist -- Colab Quickstart

**One-click pipeline: distil a Qwen3-0.6B draft from a Qwen3-4B teacher on a free T4.**

**CONFIG levels (edit in Cell 0):**  `colab_lite` (1.7B teacher, ~25 min, quick ablations) → `colab` (4B teacher, ~2-4 h, production T4 run) → `colab_a100` (8B teacher, ~2-3 h, Colab Pro/Pro+ A100)

| Cell | What it does | Time |
|------|-------------|------|
| 0. Bootstrap | **One-shot: setup + auth + run (recommended)** | starts in ~3 min |
| 1. Setup | Mount Drive, clone repo, install deps | ~3 min |
| 2. Authenticate | W&B + HuggingFace (from Colab Secrets) | ~30 s |
| 3. Run | Full pipeline (train -> merge -> eval) | ~4 h |
| 4. Resume | After session death -- re-runs from last checkpoint | ~1 min + training |
| 5. Monitor | Check progress log + state (auto-refresh option) | instant |
| 6. Dashboard | Live results dashboard in Colab tab | ~10 s |

---

### Quickest start
1. **Runtime -> Run all** (Ctrl+F9) -- Cell 0 handles everything automatically.
   Set `CONFIG`, `SMOKE`, and `BACKGROUND` at the top of Cell 0 before running.
2. To monitor while running: set `BACKGROUND = True` in Cell 0, then run Cell 5
   with `AUTO_REFRESH = True`.

### Session timeouts
- Cell 0 injects a JS keep-alive (45 s heartbeat) to prevent idle disconnects.
- Free T4 sessions are still capped at ~12 h total. For longer runs use **Kaggle**
  (30-hour sessions, free GPU, no idle timeout):
  `gbv-research/deploy/kaggle.ipynb`

### Prerequisites
1. **GPU runtime**: Runtime -> Change runtime type -> T4 GPU
2. **Colab Secrets** (left sidebar -> key icon):
   - `WANDB_API_KEY` -- from https://wandb.ai/authorize
   - `HF_TOKEN` -- from https://huggingface.co/settings/tokens
3. Google Drive mounted (Cell 0 / Cell 1 does this automatically)

In [ ]:
# =============================================================================
# Cell 0 -- ONE-SHOT BOOTSTRAP  (replaces running Cells 1-3 separately)
#
# 1. Fill in the Config section below.
# 2. Runtime -> Run all (Ctrl+F9) to run every cell automatically.
#    OR run just this cell; it handles setup + auth + pipeline in one go.
#
# BACKGROUND = True  -> pipeline runs in background, cell exits immediately;
#                        run Cell 5 with AUTO_REFRESH=True to tail the log.
# BACKGROUND = False -> output streams directly to this cell (default).
#
# CONFIG levels:
#   colab_lite  -- Qwen3-1.7B teacher (bf16, ~3.4 GB VRAM)
#                  Total VRAM: ~5.6 GB  /  T4 headroom: 9 GB
#                  300 steps, ~25 min total.  Use for quick ablations.
#   colab       -- Qwen3-4B teacher (bf16, ~8 GB VRAM)  [DEFAULT]
#                  Total VRAM: ~10.7 GB  /  T4 headroom: 4 GB
#                  500 steps, ~2-4 h total.  Production T4 run.
#   colab_a100  -- Qwen3-8B teacher (bf16, ~16 GB VRAM, Colab Pro+ only)
#                  Use colab_a100_quickstart.ipynb for that.
#
# WHY NOT 4-BIT?  4-bit NF4 quantization requires BF16 intermediate
# tensors per-weight in CPU RAM during loading.  Qwen3-8B = ~16 GB of
# intermediates -> Linux OOM killer fires (SIGKILL, exit -9, unrecoverable).
# Both 1.7B and 4B teachers load as plain BF16 -- no intermediates, no OOM.
#
# For sessions > 12 h use Kaggle (30-hour runtime, no idle timeout):
#   github.com/Rmuk655/Distill-Spec-Research/blob/main/gbv-research/deploy/kaggle.ipynb
# =============================================================================

# -- Config (edit these) ------------------------------------------------------
DRIVE_ROOT  = "/content/drive/MyDrive/specdist"
CONFIG      = "colab"    # colab_lite | colab | colab_a100 | server | laptop
SMOKE       = False      # True = smoke test (~45 min)    False = full run (~4 h)
BACKGROUND  = False      # True -> background process; monitor via Cell 5
START_DASHBOARD = False  # True -> also start Flask dashboard (Cell 6)
                         #         safe to set True even before any results exist
LOSSES      = None       # None = all losses, or e.g. "kl,ebe"
EXTRA_ARGS  = []         # e.g. ["--train_steps", "1000"]
REPO_URL    = "https://github.com/Rmuk655/Distill-Spec-Research.git"
# -----------------------------------------------------------------------------

import os, subprocess, sys, threading, time

# -- Keep-alive: prevent Colab idle-timeout while pipeline runs ---------------
# Injects a JS heartbeat (45 s) so the browser tab never goes idle.
#
# CRASH RESUME: if the session dies mid-training, just re-run this cell.
# The pipeline skips already-completed steps and the trainer auto-resumes
# from ckpt_latest (last save_every checkpoint) -- not from step 0.
# How much is lost: at most save_every steps (colab.yaml: save_every=25,
# ~1-2 min on T4 with 0.6B draft; ~2-4 min with 1.7B draft).
from IPython.display import display, Javascript
display(Javascript("""
(function() {
    if (window.__specdist_keepalive) return;
    window.__specdist_keepalive = setInterval(function() {
        var evt = new MouseEvent('mousemove', {bubbles: true});
        document.dispatchEvent(evt);
        var btn = document.querySelector('[data-tooltip="Reconnect to runtime"]');
        if (btn) btn.click();
    }, 45000);
    console.log('[specdist] keep-alive started (45 s interval)');
})();
"""))

# -- 1/5 Mount Drive ----------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(os.path.join(DRIVE_ROOT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "logs"),        exist_ok=True)
print(f"[1/5] Drive mounted  ->  {DRIVE_ROOT}")

# -- 2/5 Clone / update repo --------------------------------------------------
REPO_DIR = "/content/Distill-Spec-Research"
GBV_DIR  = f"{REPO_DIR}/gbv-research"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("[2/5] Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("[2/5] Repo updated (was already cloned)")
os.chdir(GBV_DIR)

# -- 3/5 Install deps ---------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt", "bitsandbytes", "accelerate"], check=True)
print("[3/5] Dependencies installed")

# -- 4/5 Authenticate ---------------------------------------------------------
def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

wb = _secret("WANDB_API_KEY")
if wb:
    os.environ["WANDB_API_KEY"] = wb
    import wandb; wandb.login(key=wb, relogin=True)
    print("[4/5] W&B authenticated")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("[4/5] W&B offline  (add WANDB_API_KEY via left sidebar -> Secrets)")

hf = _secret("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf
    try:
        from huggingface_hub import login
        login(token=hf, add_to_git_credential=False)
        print("     HuggingFace authenticated")
    except Exception: pass
else:
    print("     HF_TOKEN not set (OK for public Qwen3 models)")

import torch
if torch.cuda.is_available():
    free_gb = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"     GPU: {torch.cuda.get_device_name(0)}  ({free_gb:.1f} GB free)")
else:
    print("     WARNING: no GPU -- Runtime -> Change runtime type -> T4 GPU")

# -- 5/5 Run pipeline ---------------------------------------------------------
os.environ.update({
    "SPECDIST_STORAGE_ROOT": DRIVE_ROOT,
    "SPECDIST_DB_PATH":      os.path.join(DRIVE_ROOT, "results.db"),
    "SPECDIST_LOGS_ROOT":    os.path.join(DRIVE_ROOT, "logs"),
})
LOG_FILE = os.path.join(DRIVE_ROOT, "logs", "pipeline_output.log")

cmd = [sys.executable, "orchestration/experiment.py",
       "--config", CONFIG, "--storage_root", DRIVE_ROOT, "--yes"]
if SMOKE:   cmd.append("--smoke")
if LOSSES:  cmd += ["--losses", LOSSES]
cmd += EXTRA_ARGS

mode_str = "SMOKE TEST" if SMOKE else f"FULL pipeline ({CONFIG})"
print(f"\n[5/5] Launching {mode_str}")
print(f"      Log  -> {LOG_FILE}")
print(f"      DB   -> {DRIVE_ROOT}/results.db")

if BACKGROUND:
    log_fh = open(LOG_FILE, "a", buffering=1)
    _proc  = subprocess.Popen(cmd, cwd=GBV_DIR, stdout=log_fh, stderr=subprocess.STDOUT)

    def _tail():
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as lf:
            lf.seek(0, 2)
            while _proc.poll() is None:
                line = lf.readline()
                if line: sys.stdout.write(line); sys.stdout.flush()
                else:    time.sleep(0.4)
            for line in lf: sys.stdout.write(line); sys.stdout.flush()

    threading.Thread(target=_tail, daemon=True).start()
    print(f"\n  Background PID: {_proc.pid}")
    print("  Output is tee'd to the Drive log above.")
    print("  Run Cell 5 now to monitor progress without blocking.")
    print(f"  To stop: import os, signal; os.kill({_proc.pid}, signal.SIGTERM)")
# -- Optional: start Flask dashboard in background ---------------------------
if START_DASHBOARD:
    import threading, time as _t
    from google.colab.output import eval_js as _ejs
    _DASH_PORT = 5000
    os.environ["SPECDIST_DB_PATH"]   = os.path.join(DRIVE_ROOT, "results.db")
    os.environ["SPECDIST_LOGS_ROOT"] = os.path.join(DRIVE_ROOT, "logs")
    _dash_proc = [None]

    def _start_dash():
        _dash_proc[0] = subprocess.Popen(
            [sys.executable,
             os.path.join(GBV_DIR, "dashboard", "training_dashboard.py"),
             "--host", "0.0.0.0", "--port", str(_DASH_PORT)],
            env=os.environ.copy())
        _dash_proc[0].wait()

    threading.Thread(target=_start_dash, daemon=True).start()
    _t.sleep(3)  # let Flask bind the port
    _dash_url = _ejs(f"google.colab.kernel.proxyPort({_DASH_PORT})")
    print(f"\nDashboard running -> {_dash_url}")
    print("  (auto-refreshes every 15 s; new eval rows appear as they land)")

# -- Launch pipeline ----------------------------------------------------------
else:
    result = subprocess.run(cmd, cwd=GBV_DIR)
    if result.returncode == 0:
        print("\n[DONE] Pipeline complete!  Run Cell 6 for the dashboard.")
    else:
        print(f"\n[FAIL] Exit code {result.returncode} -- re-run to resume from checkpoint.")
        print(f"       Full log -> {LOG_FILE}")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 1 -- Setup
# Mount Google Drive (ALL artifacts live here -- survive session restarts)
# Clone repo to ephemeral /content/ (fast, ~5 s; re-clones each session)
# Install dependencies
# -----------------------------------------------------------------------------
import os, subprocess, sys

# -- 1a. Mount Drive ----------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# storage_root is the single directory that holds all persistent artifacts:
#   results.db, checkpoints/, logs/, pipeline_state_*.json
DRIVE_ROOT = "/content/drive/MyDrive/specdist"
os.makedirs(os.path.join(DRIVE_ROOT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, "logs"), exist_ok=True)
print(f"✓ All artifacts will persist at: {DRIVE_ROOT}")
print(f"    {DRIVE_ROOT}/results.db        ← experiment database")
print(f"    {DRIVE_ROOT}/checkpoints/      ← LoRA adapters")
print(f"    {DRIVE_ROOT}/logs/             ← pipeline + training logs")

# -- 1b. Clone repo -----------------------------------------------------------
REPO_URL  = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR  = "/content/Distill-Spec-Research"
GBV_DIR   = f"{REPO_DIR}/gbv-research"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("✓ Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo updated (already cloned)")

os.chdir(GBV_DIR)
print(f"✓ Working directory: {os.getcwd()}")

# -- 1c. Install Python dependencies -----------------------------------------
# bitsandbytes is required for 4-bit NF4 loading of the 8B teacher on a T4.
# accelerate is required by bitsandbytes device_map="auto".
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "bitsandbytes",    # 4-bit NF4 teacher
    "accelerate",      # device_map support
    "flash-attn",      # optional -- faster attention; skipped silently if compile fails
], check=False)  # flash-attn may fail on older drivers -- that's OK
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt",
    "bitsandbytes", "accelerate",
], check=True)   # re-run without flash-attn to ensure core deps are installed

print("✓ Dependencies installed")
print("\n--- Setup complete. Proceed to Cell 2 ---")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 2 -- Authenticate (W&B + HuggingFace)
#
# Reads secrets from Colab Secrets (left sidebar -> 🔑 icon).
# Required secrets:
#   WANDB_API_KEY  ->  https://wandb.ai/authorize
#   HF_TOKEN       ->  https://huggingface.co/settings/tokens
#
# If you prefer, paste keys directly below instead of using Secrets.
# -----------------------------------------------------------------------------
import os

def _get_secret(name: str, fallback: str = "") -> str:
    """Read from Colab Secrets, then env, then fallback."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name, fallback)

# -- W&B ----------------------------------------------------------------------
WANDB_API_KEY = _get_secret("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("✓ W&B authenticated")
else:
    print("⚠ WANDB_API_KEY not found -- runs will be logged offline only.")
    print("  Add it via: left sidebar -> 🔑 Secrets -> + Add new secret")
    os.environ["WANDB_MODE"] = "offline"

# -- HuggingFace --------------------------------------------------------------
HF_TOKEN = _get_secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN   # older env var
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✓ HuggingFace Hub authenticated")
    except Exception as e:
        print(f"  [HF login] {e} -- continuing anyway (public models don't need auth)")
else:
    print("ℹ HF_TOKEN not found -- OK for public models (Qwen3-0.6B, Qwen3-8B).")

# -- Confirm GPU ---------------------------------------------------------------
import torch
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}  "
          f"{free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
    if total / 1024**3 < 12:
        print("⚠ Less than 12 GB VRAM detected -- the colab config requires T4 (15 GB).")
        print("  Runtime -> Change runtime type -> T4 GPU")
else:
    print("⚠ No GPU detected -- training will be extremely slow.")
    print("  Runtime -> Change runtime type -> T4 GPU")

print("\n--- Auth complete. Proceed to Cell 3 ---")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 3 -- Run the pipeline
#
# Prerequisite: Cells 1 + 2 must have run (or use Cell 0 instead).
#
# BACKGROUND = False  -> output streams directly to this cell (default).
# BACKGROUND = True   -> pipeline runs as a background subprocess;
#                         this cell exits immediately so you can run
#                         Cell 5 with AUTO_REFRESH=True to monitor.
# -----------------------------------------------------------------------------
import os, subprocess, sys, threading, time

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# -- Configuration (edit these) -----------------------------------------------
CONFIG     = "colab"    # colab_lite | colab | colab_a100 | server | laptop
SMOKE      = False      # True = quick smoke test (~45 min)
BACKGROUND = False      # True = background process; monitor with Cell 5
LOSSES     = None       # None = all, or "kl,ebe"
EXTRA_ARGS = []         # e.g. ["--train_steps", "1000"]
# -----------------------------------------------------------------------------

if not os.path.isdir(DRIVE_ROOT):
    raise RuntimeError("DRIVE_ROOT not found -- run Cell 1 first.")

os.chdir(GBV_DIR)
os.environ.update({
    "SPECDIST_STORAGE_ROOT": DRIVE_ROOT,
    "SPECDIST_DB_PATH":      os.path.join(DRIVE_ROOT, "results.db"),
    "SPECDIST_LOGS_ROOT":    os.path.join(DRIVE_ROOT, "logs"),
})
LOG_FILE = os.path.join(DRIVE_ROOT, "logs", "pipeline_output.log")

cmd = [sys.executable, "orchestration/experiment.py",
       "--config", CONFIG, "--storage_root", DRIVE_ROOT, "--yes"]
if SMOKE:   cmd.append("--smoke")
if LOSSES:  cmd += ["--losses", LOSSES]
cmd += EXTRA_ARGS

mode_str = "SMOKE TEST" if SMOKE else f"FULL pipeline ({CONFIG})"
print(f"{'[BG] ' if BACKGROUND else ''}{mode_str}")
print(f"  Log -> {LOG_FILE}")
print(f"  DB  -> {DRIVE_ROOT}/results.db")

if BACKGROUND:
    log_fh = open(LOG_FILE, "a", buffering=1)
    _proc  = subprocess.Popen(cmd, cwd=GBV_DIR, stdout=log_fh, stderr=subprocess.STDOUT)

    def _tail():
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as lf:
            lf.seek(0, 2)
            while _proc.poll() is None:
                line = lf.readline()
                if line: sys.stdout.write(line); sys.stdout.flush()
                else:    time.sleep(0.4)
            for line in lf: sys.stdout.write(line); sys.stdout.flush()

    threading.Thread(target=_tail, daemon=True).start()
    print(f"\nBackground PID {_proc.pid} -- run Cell 5 to monitor")
    print(f"To stop: import os, signal; os.kill({_proc.pid}, signal.SIGTERM)")
else:
    result = subprocess.run(cmd, cwd=GBV_DIR)
    if result.returncode == 0:
        print("\n[DONE] Pipeline complete!  Run Cell 6 for dashboard.")
    else:
        print(f"\n[FAIL] Exit code {result.returncode} -- re-run to resume.")
        print(f"       Full log -> {LOG_FILE}")

In [ ]:
# -----------------------------------------------------------------------------
# Cell 4 -- Resume after session death
#
# Colab kills sessions after ~90 min idle (free) or ~12 h (Pro).
# Re-running this cell (or Cell 0) is all that is needed:
#   - Pipeline skips steps whose done_check file exists on Drive.
#   - Training auto-resumes from ckpt_latest (trainer crash-safe resume).
#   - At most save_every steps re-run (colab: 25 steps ~= 1-2 min).
#   - Eval steps pass --skip_existing so completed cells are skipped.
# -----------------------------------------------------------------------------
import os, subprocess, sys

# -- Re-mount Drive and re-clone if needed (idempotent) -----------------------
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"
DRIVE_ROOT = "/content/drive/MyDrive/specdist"   # same as Cell 1 & 3

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", f"{GBV_DIR}/requirements.txt",
                    "bitsandbytes", "accelerate"], check=True)
    print("✓ Repo re-cloned, deps re-installed")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo still present -- updated")

os.chdir(GBV_DIR)

# -- Re-authenticate W&B -------------------------------------------------------
try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
        import wandb; wandb.login(key=key, relogin=True)
        print("✓ W&B re-authenticated")
except Exception:
    os.environ.setdefault("WANDB_MODE", "offline")
    print("⚠ W&B key not found -- logging offline")

# -- Re-set storage env vars ---------------------------------------------------
os.environ["SPECDIST_STORAGE_ROOT"] = DRIVE_ROOT
os.environ["SPECDIST_DB_PATH"]       = os.path.join(DRIVE_ROOT, "results.db")
os.environ["SPECDIST_LOGS_ROOT"]     = os.path.join(DRIVE_ROOT, "logs")

# -- Resume pipeline -----------------------------------------------------------
CONFIG = "colab"   # must match what was used in Cell 3 (colab_lite | colab)

print(f"\nResuming pipeline (config={CONFIG}, storage_root={DRIVE_ROOT})")
print("Pipeline will skip already-completed steps and resume from last failure.\n")

subprocess.run([
    sys.executable, "orchestration/experiment.py",
    "--config",       CONFIG,
    "--storage_root", DRIVE_ROOT,
    "--yes",
], cwd=GBV_DIR)

In [ ]:
# -----------------------------------------------------------------------------
# Cell 5 -- Monitor progress
#
# Safe to run any time, including while Cell 0 / Cell 3 is running.
# AUTO_REFRESH = True  -> re-prints every REFRESH_SECS seconds (live tail).
#                         Interrupt the cell (square button) to stop.
# AUTO_REFRESH = False -> single snapshot (default).
# -----------------------------------------------------------------------------
import os, json, glob, datetime, time
from IPython.display import clear_output

DRIVE_ROOT   = "/content/drive/MyDrive/specdist"
LOG_TAIL     = 60      # log lines to show
AUTO_REFRESH = False   # True = live tail loop
REFRESH_SECS = 20      # seconds between refreshes

LOG_FILE   = f"{DRIVE_ROOT}/logs/pipeline_output.log"
STATE_FILE = f"{DRIVE_ROOT}/pipeline_state_colab.json"

def _show():
    sep = "=" * 62
    # -- 1. Pipeline state ----------------------------------------------------
    print(sep)
    print("PIPELINE STATE")
    print(sep)
    if os.path.exists(STATE_FILE):
        state = json.load(open(STATE_FILE, encoding="utf-8"))
        steps  = state.get("steps", {})
        counts = {"done": 0, "running": 0, "pending": 0, "error": 0}
        for sid, info in steps.items():
            s    = info.get("status", "pending")
            counts[s] = counts.get(s, 0) + 1
            icon = {"done": "OK", "running": ">>", "error": "!!", "pending": ".."}.get(s, "..")
            ts   = (info.get("finished_at") or info.get("started_at") or "")[:16]
            print(f"  [{icon}] {sid:45s}  {s:8s}  {ts}")
        print(f"\n  Done: {counts['done']}  Running: {counts['running']}"
              f"  Pending: {counts.get('pending',0)}  Error: {counts['error']}")
    else:
        print(f"  State file not found: {STATE_FILE}")
        print("  Pipeline has not started yet (or Drive is not mounted).")

    # -- 2. Log tail ----------------------------------------------------------
    print()
    print(sep)
    print(f"PIPELINE LOG (last {LOG_TAIL} lines)")
    print(sep)
    if os.path.exists(LOG_FILE):
        lines = open(LOG_FILE, encoding="utf-8", errors="replace").readlines()
        print("".join(lines[-LOG_TAIL:]))
        age_s  = datetime.datetime.now().timestamp() - os.path.getmtime(LOG_FILE)
        status = "ACTIVE" if age_s < 120 else f"STALE ({int(age_s//60)} min ago)"
        print(f"[{len(lines)} lines | {status}]")
    else:
        print(f"  Not found: {LOG_FILE}")

    # -- 3. Error snapshots ---------------------------------------------------
    err_logs = sorted(glob.glob(f"{DRIVE_ROOT}/logs/step_*_error.log"))
    if err_logs:
        print()
        print(sep)
        print(f"ERROR SNAPSHOTS ({len(err_logs)} file(s))")
        print(sep)
        for f in err_logs:
            print(f"\n--- {os.path.basename(f)} ---")
            txt = open(f, encoding="utf-8", errors="replace").read()
            print(txt[-2000:] if len(txt) > 2000 else txt)

    if AUTO_REFRESH:
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"\n[{ts}  |  next refresh in {REFRESH_SECS}s  |  interrupt to stop]")

if AUTO_REFRESH:
    print(f"Auto-refresh every {REFRESH_SECS}s -- interrupt cell (square button) to stop.")
    while True:
        clear_output(wait=True)
        _show()
        time.sleep(REFRESH_SECS)
else:
    _show()

In [ ]:
# -----------------------------------------------------------------------------
# Cell 6 -- Live results dashboard
#
# Starts the Flask dashboard server bound to 0.0.0.0 so Colab's port proxy
# can tunnel it to a public HTTPS URL you can open in any browser tab.
#
# Safe to run while Cell 3 is training -- the dashboard reads the DB live
# and shows new eval rows as they land.
#
# Shows:
#   • Block efficiency heatmap (loss × verifier)
#   • Training loss curves
#   • Per-step acceptance-rate breakdown
#   • Live pipeline log tail
#
# To stop: Runtime -> Interrupt execution  (or just close the tab -- server
# keeps running in background until the Colab session dies)
# -----------------------------------------------------------------------------
import os, sys, time, threading, subprocess
from google.colab.output import eval_js

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"
PORT       = 5000

# Point dashboard at Drive DB + logs (not ephemeral /content/)
os.environ["SPECDIST_DB_PATH"]   = f"{DRIVE_ROOT}/results.db"
os.environ["SPECDIST_LOGS_ROOT"] = f"{DRIVE_ROOT}/logs"

if not os.path.exists(os.environ["SPECDIST_DB_PATH"]):
    print(f"⚠ No results DB yet at {os.environ['SPECDIST_DB_PATH']}")
    print("  Run Cell 3 first to generate some results, then come back here.")
else:
    # Launch Flask server in a background thread so this cell doesn't block
    _server_proc = [None]

    def _start_server():
        _server_proc[0] = subprocess.Popen(
            [
                sys.executable,
                f"{GBV_DIR}/dashboard/training_dashboard.py",
                "--host", "0.0.0.0",
                "--port", str(PORT),
            ],
            env=os.environ.copy(),
        )
        _server_proc[0].wait()

    t = threading.Thread(target=_start_server, daemon=True)
    t.start()

    # Give Flask a moment to bind the port
    time.sleep(3)

    # Get the Colab-proxied public URL for this port
    public_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")

    print(f"✓ Dashboard running")
    print(f"  Open this URL in any browser tab:")
    print(f"  {public_url}")
    print()
    print(f"  DB   : {os.environ['SPECDIST_DB_PATH']}")
    print(f"  Logs : {os.environ['SPECDIST_LOGS_ROOT']}")
    print(f"  Port : {PORT} (proxied by Colab to HTTPS above)")
    print()
    print("  The dashboard auto-refreshes every 15 s -- new eval rows appear as they land.")
    print("  To stop the server: Runtime -> Interrupt execution")